# High-ROI Model Improvements: Multi-Label Heads, Intensity Regression, Better Evaluation

This notebook is a starter implementation for the highest-ROI changes suggested by the paper review:

- multi-label emotion/state heads instead of mutually exclusive labels
- item/intensity regression heads for graded psychological constructs
- validation-tuned thresholds and clinically/practically meaningful evaluation

It is designed to complement `llm_model_comparison_baseline.ipynb`. That existing notebook evaluates LLM soft labels for reappraisal strategies. This notebook sketches a trainable multitask model and a stronger evaluation protocol that can be adapted once emotion/state/intensity labels are available.

Recommended packages for the full training path: `torch`, `transformers`, `scikit-learn`, and optionally `iterative-stratification` for multilabel stratified splits.


## 0. What Has To Be Considered

Before implementing these heads in the real model, decide the following explicitly:

1. **Target definitions**: emotion/state labels must be non-exclusive; intensity items must be ordinal or continuous and tied to a clear construct.
2. **Annotation source**: human labels are preferred. Lexicon or LLM labels can be useful as weak labels, but should not be treated as clinical ground truth.
3. **Data leakage**: split by participant, prompt/item family, session, or source when possible. Random row splits can overstate generalization.
4. **Class imbalance**: rare labels need `pos_weight`, focal loss, oversampling, or targeted collection. Always report per-label metrics.
5. **Thresholding**: do not hard-code `0.5` as the final decision threshold. Tune thresholds on validation only, then freeze for test.
6. **Regression evaluation**: use MAE/RMSE, item-wise errors, and tolerance metrics such as within `+/- 1` when labels are ordinal/noisy.
7. **Calibration**: predicted probabilities and intensities should be reliable, not merely high-scoring.
8. **Safety-sensitive labels**: evaluate false negatives and false positives separately for risk-related states; aggregate F1 is not enough.
9. **Ablations**: compare shared encoder only, encoder + multi-label heads, encoder + regression heads, and full multitask setup.
10. **Human evaluation**: for generated reappraisals, add human ratings of empathy, specificity, plausibility, helpfulness, and safety.


In [ ]:
from pathlib import Path
import json
import math
import os
import random
import warnings

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

try:
    import torch
    from torch import nn
    from torch.utils.data import Dataset, DataLoader
    TORCH_AVAILABLE = True
except ImportError:
    torch = None
    nn = None
    DataLoader = None
    TORCH_AVAILABLE = False

    class Dataset:
        pass

try:
    from sklearn.metrics import (
        average_precision_score,
        brier_score_loss,
        classification_report,
        f1_score,
        hamming_loss,
        jaccard_score,
        mean_absolute_error,
        mean_squared_error,
        precision_recall_fscore_support,
        roc_auc_score,
    )
    from sklearn.model_selection import GroupShuffleSplit, train_test_split
    EVAL_AVAILABLE = True
except ImportError:
    average_precision_score = None
    brier_score_loss = None
    classification_report = None
    f1_score = None
    hamming_loss = None
    jaccard_score = None
    mean_absolute_error = None
    mean_squared_error = None
    precision_recall_fscore_support = None
    roc_auc_score = None
    GroupShuffleSplit = None
    train_test_split = None
    EVAL_AVAILABLE = False

try:
    from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
except ImportError:
    AutoModel = None
    AutoTokenizer = None
    get_linear_schedule_with_warmup = None

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
except ImportError:
    MultilabelStratifiedShuffleSplit = None

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
if TORCH_AVAILABLE:
    torch.manual_seed(RANDOM_STATE)

DEVICE = 'cuda' if TORCH_AVAILABLE and torch.cuda.is_available() else 'mps' if TORCH_AVAILABLE and torch.backends.mps.is_available() else 'cpu'
DEVICE


## 1. Configuration

The existing baseline notebook appears to use eight reappraisal strategy labels from annotator vote fractions. The new emotion/state and intensity columns are placeholders until you add labels.

Recommended data shape:

- `SITUATION`: original situation text
- `REAPPRAISAL`: candidate reappraisal text
- one column per multi-label target in `[0, 1]`
- one column per intensity target as a continuous or ordinal value, e.g. `0..6` or `0..1`
- optional `participant_id`, `session_id`, `source_id`, or `item_family` for leakage-safe splits


In [ ]:
# Locate the consolidated project from either the project root or a nested notebook folder.
def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'data' / 'annotate-lime.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root containing data/annotate-lime.csv.')

PROJECT_DIR = Path(os.getenv('REAPPRAISAL_PROJECT_DIR') or find_project_root()).resolve()
DATA_DIR = Path(os.getenv('REAPPRAISAL_DATA_DIR') or PROJECT_DIR / 'data').resolve()
OUTPUT_DIR = Path(os.getenv('REAPPRAISAL_MULTITASK_OUTPUT_DIR') or PROJECT_DIR / 'Code' / 'multitask' / 'outputs').resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COLS = ['SITUATION', 'REAPPRAISAL']
GROUP_COL_CANDIDATES = ['participant_id', 'session_id', 'source_id', 'item_family', 'key']

REAPPRAISAL_STRATEGY_LABELS = ['GM', 'PR', 'A', 'EJ', 'N', 'UC', 'V', 'H']
STRATEGY_LABEL_NAMES = {
    'GM': 'growth mindset',
    'PR': 'positive repurposing',
    'A': 'aberration',
    'EJ': 'external justification',
    'N': 'normalizing',
    'UC': 'unwarranted conclusions',
    'V': 'validation',
    'H': 'humor',
}

# Replace or extend these once you have labels. Keep them multi-label.
EMOTION_STATE_LABELS = [
    'anger',
    'sadness',
    'fear',
    'anxiety',
    'shame',
    'hopelessness',
    'loneliness',
    'self_blame',
    'validation_need',
]

# Replace or extend these once you have graded labels.
INTENSITY_TARGETS = [
    'distress_intensity',
    'rumination_intensity',
    'reappraisal_helpfulness',
]

MULTILABEL_TARGETS = REAPPRAISAL_STRATEGY_LABELS + EMOTION_STATE_LABELS
LABEL_THRESHOLD = 0.60
MAX_LENGTH = 256
BASE_MODEL = 'microsoft/deberta-v3-base'
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRAD_CLIP_NORM = 1.0
MULTILABEL_LOSS_WEIGHT = 1.0
REGRESSION_LOSS_WEIGHT = 0.5
RUN_TRAINING = False

config = {
    'base_model': BASE_MODEL,
    'multilabel_targets': MULTILABEL_TARGETS,
    'intensity_targets': INTENSITY_TARGETS,
    'label_threshold': LABEL_THRESHOLD,
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO,
}
config


## 2. Load Existing Reappraisal Strategy Labels

This mirrors the current baseline notebook: each annotation file is an annotator-by-item matrix, and the soft target is the fraction of annotators who marked a strategy present.

If your dataset is somewhere else, set `REAPPRAISAL_DATA_DIR` before running the notebook.


In [ ]:
def load_reappraisal_strategy_dataset(data_dir: Path) -> pd.DataFrame:
    texts_path = data_dir / 'annotate-lime.csv'
    if not texts_path.exists():
        raise FileNotFoundError(
            f'Missing {texts_path}. Set REAPPRAISAL_DATA_DIR to the folder containing annotate-lime.csv.'
        )

    texts = pd.read_csv(texts_path)
    missing_text_cols = [col for col in TEXT_COLS if col not in texts.columns]
    if missing_text_cols:
        raise ValueError(f'Missing required text columns: {missing_text_cols}')

    soft_labels = pd.DataFrame()
    positive_votes = pd.DataFrame()
    vote_counts = pd.DataFrame()

    for label in REAPPRAISAL_STRATEGY_LABELS:
        ann_path = data_dir / f'annotations-only-[{label}].csv'
        if not ann_path.exists():
            raise FileNotFoundError(f'Missing annotation file: {ann_path}')
        ann = pd.read_csv(ann_path, sep=';').apply(pd.to_numeric, errors='coerce')
        positive_votes[f'{label}_positive_votes'] = ann.sum(axis=0).values
        vote_counts[f'{label}_n_votes'] = ann.count(axis=0).values
        soft_labels[label] = ann.mean(axis=0).values

    dataset = pd.concat(
        [
            texts.reset_index(drop=True),
            soft_labels.reset_index(drop=True),
            positive_votes.reset_index(drop=True),
            vote_counts.reset_index(drop=True),
        ],
        axis=1,
    )
    if 'question_code' not in dataset.columns:
        dataset.insert(0, 'question_code', [f'Q{i + 1}' for i in range(len(dataset))])
    return dataset

try:
    dataset = load_reappraisal_strategy_dataset(DATA_DIR)
    print('Loaded dataset:', dataset.shape)
    display(dataset.head())
except Exception as exc:
    dataset = pd.DataFrame()
    print('Dataset not loaded yet:', exc)


## 3. Attach Emotion/State And Intensity Targets

Use this section once you have new labels. The function expects one row per `key` or `question_code` with any subset of `EMOTION_STATE_LABELS` and `INTENSITY_TARGETS`.

Important: keep missing targets as `NaN`. The training loss below uses masks so rows without a particular target do not contribute to that head.


In [ ]:
def attach_optional_targets(
    base_df: pd.DataFrame,
    labels_path: Path | None = None,
    join_keys=('question_code', 'key'),
) -> pd.DataFrame:
    df = base_df.copy()
    if labels_path is not None and labels_path.exists():
        extra = pd.read_csv(labels_path)
        possible_keys = [key for key in join_keys if key in df.columns and key in extra.columns]
        if not possible_keys:
            raise ValueError(f'No shared join key found. Tried: {join_keys}')
        df = df.merge(extra, on=possible_keys, how='left', suffixes=('', '_extra'))

    for col in EMOTION_STATE_LABELS:
        if col not in df.columns:
            df[col] = np.nan
    for col in INTENSITY_TARGETS:
        if col not in df.columns:
            df[col] = np.nan
    return df

# Example: put your new labels here when ready.
OPTIONAL_LABELS_PATH = Path(os.getenv('REAPPRAISAL_EXTRA_LABELS_CSV', '')) if os.getenv('REAPPRAISAL_EXTRA_LABELS_CSV') else None

if not dataset.empty:
    dataset = attach_optional_targets(dataset, OPTIONAL_LABELS_PATH)
    display(dataset[TEXT_COLS + REAPPRAISAL_STRATEGY_LABELS + EMOTION_STATE_LABELS + INTENSITY_TARGETS].head())


## 4. Data Audit

Run this before training. The goal is to find target sparsity, imbalance, impossible values, and split risks early.


In [ ]:
def audit_targets(df: pd.DataFrame, multilabel_cols, regression_cols, threshold=LABEL_THRESHOLD):
    rows = []
    for col in multilabel_cols:
        if col not in df.columns:
            rows.append({'target': col, 'type': 'multi-label', 'present': False})
            continue
        series = pd.to_numeric(df[col], errors='coerce')
        valid = series.notna()
        binary = series[valid] >= threshold
        rows.append({
            'target': col,
            'type': 'multi-label',
            'present': True,
            'n_valid': int(valid.sum()),
            'min': float(series[valid].min()) if valid.any() else np.nan,
            'max': float(series[valid].max()) if valid.any() else np.nan,
            'positive_rate_at_threshold': float(binary.mean()) if valid.any() else np.nan,
        })
    for col in regression_cols:
        if col not in df.columns:
            rows.append({'target': col, 'type': 'regression', 'present': False})
            continue
        series = pd.to_numeric(df[col], errors='coerce')
        valid = series.notna()
        rows.append({
            'target': col,
            'type': 'regression',
            'present': True,
            'n_valid': int(valid.sum()),
            'min': float(series[valid].min()) if valid.any() else np.nan,
            'max': float(series[valid].max()) if valid.any() else np.nan,
            'mean': float(series[valid].mean()) if valid.any() else np.nan,
            'std': float(series[valid].std()) if valid.sum() > 1 else np.nan,
        })
    return pd.DataFrame(rows)

if not dataset.empty:
    audit_df = audit_targets(dataset, MULTILABEL_TARGETS, INTENSITY_TARGETS)
    display(audit_df)
    audit_df.to_csv(OUTPUT_DIR / 'target_audit.csv', index=False)


## 5. Leakage-Aware Splits

Prefer grouped splits when there is any repeated participant, session, scenario, or item family. If no group column exists, this falls back to multilabel stratification when available, otherwise a random split.


In [ ]:
def choose_group_col(df: pd.DataFrame):
    for col in GROUP_COL_CANDIDATES:
        if col in df.columns and df[col].notna().any() and df[col].nunique() < len(df):
            return col
    return None


def multilabel_or_random_split(df, test_size, random_state):
    y_cols = [col for col in MULTILABEL_TARGETS if col in df.columns and df[col].notna().any()]
    if y_cols and MultilabelStratifiedShuffleSplit is not None:
        y = (df[y_cols].fillna(0).to_numpy(dtype=float) >= LABEL_THRESHOLD).astype(int)
        splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        left_idx, right_idx = next(splitter.split(df, y))
        return df.iloc[left_idx].reset_index(drop=True), df.iloc[right_idx].reset_index(drop=True)
    if train_test_split is not None:
        return train_test_split(df, test_size=test_size, random_state=random_state, shuffle=True)
    rng = np.random.default_rng(random_state)
    indices = rng.permutation(len(df))
    n_right = int(round(len(df) * test_size))
    right_idx = indices[:n_right]
    left_idx = indices[n_right:]
    return df.iloc[left_idx].reset_index(drop=True), df.iloc[right_idx].reset_index(drop=True)


def make_splits(df: pd.DataFrame):
    group_col = choose_group_col(df)
    if group_col is not None and GroupShuffleSplit is not None:
        gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
        train_idx, temp_idx = next(gss.split(df, groups=df[group_col]))
        train = df.iloc[train_idx].reset_index(drop=True)
        temp = df.iloc[temp_idx].reset_index(drop=True)
        gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
        val_idx, test_idx = next(gss2.split(temp, groups=temp[group_col]))
        val = temp.iloc[val_idx].reset_index(drop=True)
        test = temp.iloc[test_idx].reset_index(drop=True)
        split_kind = f'grouped by {group_col}'
    else:
        train, temp = multilabel_or_random_split(df, test_size=0.30, random_state=RANDOM_STATE)
        val, test = multilabel_or_random_split(temp, test_size=0.50, random_state=RANDOM_STATE)
        split_kind = 'multilabel stratified if iterstrat is installed, otherwise random'
    return {'train': train, 'validation': val, 'test': test}, split_kind

if not dataset.empty:
    split_frames, split_kind = make_splits(dataset)
    print('Split kind:', split_kind)
    for split_name, split_df in split_frames.items():
        print(split_name, split_df.shape)


## 6. Dataset And Collator

The model input uses both situation and reappraisal so the encoder can judge the emotional context and the strategy response together.


In [ ]:
class ReappraisalMultitaskDataset(Dataset):
    def __init__(self, df, tokenizer, multilabel_cols, regression_cols, max_length=MAX_LENGTH):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.multilabel_cols = multilabel_cols
        self.regression_cols = regression_cols
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = f'Situation: {row["SITUATION"]}\nReappraisal: {row["REAPPRAISAL"]}'
        enc = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {key: value.squeeze(0) for key, value in enc.items()}

        multilabel_values = pd.to_numeric(row[self.multilabel_cols], errors='coerce').to_numpy(dtype=np.float32)
        regression_values = pd.to_numeric(row[self.regression_cols], errors='coerce').to_numpy(dtype=np.float32)
        item['multilabel_targets'] = torch.tensor(np.nan_to_num(multilabel_values, nan=0.0), dtype=torch.float32)
        item['multilabel_mask'] = torch.tensor(~np.isnan(multilabel_values), dtype=torch.float32)
        item['regression_targets'] = torch.tensor(np.nan_to_num(regression_values, nan=0.0), dtype=torch.float32)
        item['regression_mask'] = torch.tensor(~np.isnan(regression_values), dtype=torch.float32)
        return item


## 7. Multitask Model: Shared Encoder + Multi-Label + Regression Heads

This follows the high-ROI pattern from the papers: one shared language encoder, independent sigmoid outputs for overlapping states/strategies, and separate continuous outputs for item/intensity targets.


In [ ]:
BaseModule = nn.Module if TORCH_AVAILABLE else object


class ReappraisalMultiTaskModel(BaseModule):
    def __init__(self, model_name, n_multilabel, n_regression, dropout=0.2):
        if not TORCH_AVAILABLE:
            raise ImportError('Install PyTorch before creating the model.')
        super().__init__()
        if AutoModel is None:
            raise ImportError('Install transformers before creating the model.')
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.multilabel_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, n_multilabel),
        )
        self.regression_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, n_regression),
        )

    def pooled_output(self, outputs):
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            return outputs.pooler_output
        return outputs.last_hidden_state[:, 0]

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if token_type_ids is not None:
            kwargs['token_type_ids'] = token_type_ids
        outputs = self.encoder(**kwargs)
        pooled = self.dropout(self.pooled_output(outputs))
        return {
            'multilabel_logits': self.multilabel_head(pooled),
            'regression_values': self.regression_head(pooled),
        }


## 8. Loss Functions

Key details:

- `BCEWithLogitsLoss` supports independent labels.
- `pos_weight` counters imbalance for each multi-label target.
- masked losses let you add emotion/intensity labels gradually without throwing away rows.
- `SmoothL1Loss` is often gentler than MSE for noisy ordinal intensity labels.


In [ ]:
def compute_pos_weight(df, multilabel_cols, threshold=LABEL_THRESHOLD, clip_max=20.0):
    weights = []
    for col in multilabel_cols:
        series = pd.to_numeric(df[col], errors='coerce')
        valid = series.notna()
        if not valid.any():
            weights.append(1.0)
            continue
        positives = float((series[valid] >= threshold).sum())
        negatives = float(valid.sum() - positives)
        if positives <= 0:
            weights.append(1.0)
        else:
            weights.append(min(negatives / positives, clip_max))
    return torch.tensor(weights, dtype=torch.float32)


def masked_bce_with_logits(logits, targets, mask, pos_weight=None):
    loss = nn.functional.binary_cross_entropy_with_logits(
        logits,
        targets,
        pos_weight=pos_weight,
        reduction='none',
    )
    loss = loss * mask
    denom = mask.sum().clamp_min(1.0)
    return loss.sum() / denom


def masked_smooth_l1(pred, targets, mask):
    loss = nn.functional.smooth_l1_loss(pred, targets, reduction='none')
    loss = loss * mask
    denom = mask.sum().clamp_min(1.0)
    return loss.sum() / denom


def compute_loss(outputs, batch, pos_weight=None):
    multilabel_loss = masked_bce_with_logits(
        outputs['multilabel_logits'],
        batch['multilabel_targets'],
        batch['multilabel_mask'],
        pos_weight=pos_weight,
    )
    regression_loss = masked_smooth_l1(
        outputs['regression_values'],
        batch['regression_targets'],
        batch['regression_mask'],
    )
    total = MULTILABEL_LOSS_WEIGHT * multilabel_loss + REGRESSION_LOSS_WEIGHT * regression_loss
    return total, {'multilabel_loss': multilabel_loss.item(), 'regression_loss': regression_loss.item()}


## 9. Training Loop

Leave `RUN_TRAINING = False` until your extra labels are present and you have checked the target audit.


In [ ]:
def move_batch_to_device(batch, device):
    return {key: value.to(device) if torch.is_tensor(value) else value for key, value in batch.items()}


def train_one_epoch(model, loader, optimizer, scheduler, pos_weight=None):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = move_batch_to_device(batch, DEVICE)
        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch.get('attention_mask'),
            token_type_ids=batch.get('token_type_ids'),
        )
        loss, _ = compute_loss(outputs, batch, pos_weight=pos_weight)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)


def fit_model(split_frames):
    if not TORCH_AVAILABLE:
        raise ImportError('Install PyTorch before training.')
    if AutoTokenizer is None:
        raise ImportError('Install transformers before training.')
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    train_ds = ReappraisalMultitaskDataset(split_frames['train'], tokenizer, MULTILABEL_TARGETS, INTENSITY_TARGETS)
    val_ds = ReappraisalMultitaskDataset(split_frames['validation'], tokenizer, MULTILABEL_TARGETS, INTENSITY_TARGETS)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = ReappraisalMultiTaskModel(BASE_MODEL, len(MULTILABEL_TARGETS), len(INTENSITY_TARGETS)).to(DEVICE)
    pos_weight = compute_pos_weight(split_frames['train'], MULTILABEL_TARGETS).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps,
    )

    history = []
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, pos_weight=pos_weight)
        val_predictions = predict(model, val_loader)
        val_summary = summarize_predictions(val_predictions, MULTILABEL_TARGETS, INTENSITY_TARGETS)
        row = {'epoch': epoch, 'train_loss': train_loss, **val_summary}
        history.append(row)
        print(row)
    return model, tokenizer, pd.DataFrame(history)

if RUN_TRAINING and not dataset.empty:
    model, tokenizer, history_df = fit_model(split_frames)
    history_df.to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
else:
    print('Training skipped. Set RUN_TRAINING=True after adding/checking labels.')


## 10. Prediction Helpers


In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    rows = []
    for batch in loader:
        batch = move_batch_to_device(batch, DEVICE)
        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch.get('attention_mask'),
            token_type_ids=batch.get('token_type_ids'),
        )
        rows.append({
            'multilabel_probs': torch.sigmoid(outputs['multilabel_logits']).detach().cpu().numpy(),
            'multilabel_targets': batch['multilabel_targets'].detach().cpu().numpy(),
            'multilabel_mask': batch['multilabel_mask'].detach().cpu().numpy(),
            'regression_values': outputs['regression_values'].detach().cpu().numpy(),
            'regression_targets': batch['regression_targets'].detach().cpu().numpy(),
            'regression_mask': batch['regression_mask'].detach().cpu().numpy(),
        })
    return {
        key: np.concatenate([row[key] for row in rows], axis=0)
        for key in rows[0]
    } if rows else {}


def summarize_predictions(predictions, multilabel_cols, regression_cols):
    if not predictions:
        return {}
    metrics = evaluate_multilabel_predictions(
        predictions['multilabel_targets'],
        predictions['multilabel_probs'],
        predictions['multilabel_mask'],
        multilabel_cols,
        thresholds={label: 0.5 for label in multilabel_cols},
    )
    reg = evaluate_regression_predictions(
        predictions['regression_targets'],
        predictions['regression_values'],
        predictions['regression_mask'],
        regression_cols,
    )
    return {
        'macro_f1_0_5': metrics['overall']['macro_f1'],
        'micro_f1_0_5': metrics['overall']['micro_f1'],
        'mean_regression_mae': reg['overall'].get('mean_mae', np.nan),
    }


## 11. Better Evaluation: Multi-Label Metrics

Report both soft-score quality and binary decision quality. Tune binary thresholds on validation only.


In [ ]:
def tune_thresholds(y_true_soft, y_score, mask, label_cols, target_threshold=LABEL_THRESHOLD, grid=None):
    if not EVAL_AVAILABLE:
        raise ImportError('Install scikit-learn before tuning thresholds.')
    if grid is None:
        grid = np.round(np.linspace(0.05, 0.95, 19), 2)
    thresholds = {}
    rows = []
    for i, label in enumerate(label_cols):
        valid = mask[:, i].astype(bool)
        if valid.sum() == 0:
            thresholds[label] = 0.5
            rows.append({'label': label, 'threshold': 0.5, 'validation_f1': np.nan, 'n_valid': 0})
            continue
        y_bin = (y_true_soft[valid, i] >= target_threshold).astype(int)
        scores = y_score[valid, i]
        best_threshold, best_f1 = 0.5, -1.0
        for threshold in grid:
            pred = (scores >= threshold).astype(int)
            score = f1_score(y_bin, pred, zero_division=0)
            if score > best_f1:
                best_threshold, best_f1 = float(threshold), float(score)
        thresholds[label] = best_threshold
        rows.append({'label': label, 'threshold': best_threshold, 'validation_f1': best_f1, 'n_valid': int(valid.sum())})
    return thresholds, pd.DataFrame(rows)


def evaluate_multilabel_predictions(y_true_soft, y_score, mask, label_cols, thresholds):
    if not EVAL_AVAILABLE:
        raise ImportError('Install scikit-learn before evaluating multi-label predictions.')
    y_true_bin = (y_true_soft >= LABEL_THRESHOLD).astype(int)
    threshold_arr = np.array([thresholds.get(label, 0.5) for label in label_cols])
    y_pred_bin = (y_score >= threshold_arr).astype(int)

    label_rows = []
    valid_any = mask.astype(bool)
    for i, label in enumerate(label_cols):
        valid = valid_any[:, i]
        if valid.sum() == 0:
            continue
        yt = y_true_bin[valid, i]
        yp = y_pred_bin[valid, i]
        ys = y_score[valid, i]
        precision, recall, f1, support = precision_recall_fscore_support(
            yt, yp, average='binary', zero_division=0
        )
        row = {
            'label': label,
            'n_valid': int(valid.sum()),
            'positive_rate': float(yt.mean()),
            'threshold': thresholds.get(label, 0.5),
            'precision': float(precision),
            'recall': float(recall),
            'f1': float(f1),
            'brier': float(brier_score_loss(yt, ys)) if len(np.unique(yt)) > 1 else np.nan,
        }
        if len(np.unique(yt)) > 1:
            row['average_precision'] = float(average_precision_score(yt, ys))
            row['roc_auc'] = float(roc_auc_score(yt, ys))
        else:
            row['average_precision'] = np.nan
            row['roc_auc'] = np.nan
        label_rows.append(row)

    full_valid_rows = valid_any.all(axis=1)
    if full_valid_rows.any():
        yt_full = y_true_bin[full_valid_rows]
        yp_full = y_pred_bin[full_valid_rows]
        overall = {
            'micro_f1': float(f1_score(yt_full, yp_full, average='micro', zero_division=0)),
            'macro_f1': float(f1_score(yt_full, yp_full, average='macro', zero_division=0)),
            'samples_f1': float(f1_score(yt_full, yp_full, average='samples', zero_division=0)),
            'jaccard_samples': float(jaccard_score(yt_full, yp_full, average='samples', zero_division=0)),
            'hamming_loss': float(hamming_loss(yt_full, yp_full)),
            'n_full_valid_rows': int(full_valid_rows.sum()),
        }
    else:
        overall = {'n_full_valid_rows': 0}

    return {'overall': overall, 'per_label': pd.DataFrame(label_rows)}


## 12. Better Evaluation: Intensity Regression Metrics

Use item-wise MAE/RMSE and tolerance accuracy. The `within_1` metric is useful when labels are ordinal and small errors are acceptable, similar to symptom rating scales.


In [ ]:
def evaluate_regression_predictions(y_true, y_pred, mask, target_cols, tolerance=1.0):
    if not EVAL_AVAILABLE:
        raise ImportError('Install scikit-learn before evaluating regression predictions.')
    rows = []
    for i, target in enumerate(target_cols):
        valid = mask[:, i].astype(bool)
        if valid.sum() == 0:
            continue
        yt = y_true[valid, i]
        yp = y_pred[valid, i]
        abs_error = np.abs(yt - yp)
        rows.append({
            'target': target,
            'n_valid': int(valid.sum()),
            'mae': float(mean_absolute_error(yt, yp)),
            'rmse': float(math.sqrt(mean_squared_error(yt, yp))),
            'within_tolerance': float((abs_error <= tolerance).mean()),
            'mean_true': float(np.mean(yt)),
            'mean_pred': float(np.mean(yp)),
            'bias_mean_pred_minus_true': float(np.mean(yp - yt)),
        })
    per_item = pd.DataFrame(rows)
    overall = {
        'mean_mae': float(per_item['mae'].mean()) if not per_item.empty else np.nan,
        'mean_rmse': float(per_item['rmse'].mean()) if not per_item.empty else np.nan,
        'mean_within_tolerance': float(per_item['within_tolerance'].mean()) if not per_item.empty else np.nan,
    }
    return {'overall': overall, 'per_item': per_item}


## 13. Validation-Tuned Final Evaluation

Use this after training or after loading saved predictions. The test set should be touched only once thresholds are frozen.


In [ ]:
def final_evaluate(val_predictions, test_predictions):
    thresholds, threshold_df = tune_thresholds(
        val_predictions['multilabel_targets'],
        val_predictions['multilabel_probs'],
        val_predictions['multilabel_mask'],
        MULTILABEL_TARGETS,
    )
    threshold_df.to_csv(OUTPUT_DIR / 'validation_tuned_thresholds.csv', index=False)

    multi = evaluate_multilabel_predictions(
        test_predictions['multilabel_targets'],
        test_predictions['multilabel_probs'],
        test_predictions['multilabel_mask'],
        MULTILABEL_TARGETS,
        thresholds,
    )
    reg = evaluate_regression_predictions(
        test_predictions['regression_targets'],
        test_predictions['regression_values'],
        test_predictions['regression_mask'],
        INTENSITY_TARGETS,
    )

    multi['per_label'].to_csv(OUTPUT_DIR / 'test_multilabel_per_label_metrics.csv', index=False)
    reg['per_item'].to_csv(OUTPUT_DIR / 'test_regression_per_item_metrics.csv', index=False)
    with open(OUTPUT_DIR / 'test_overall_metrics.json', 'w') as f:
        json.dump({'multilabel': multi['overall'], 'regression': reg['overall']}, f, indent=2)

    print('Multi-label overall')
    display(pd.DataFrame([multi['overall']]))
    print('Multi-label per label')
    display(multi['per_label'])
    print('Regression overall')
    display(pd.DataFrame([reg['overall']]))
    print('Regression per item')
    display(reg['per_item'])
    return thresholds, multi, reg

# Example after training:
# val_loader = DataLoader(ReappraisalMultitaskDataset(split_frames['validation'], tokenizer, MULTILABEL_TARGETS, INTENSITY_TARGETS), batch_size=BATCH_SIZE)
# test_loader = DataLoader(ReappraisalMultitaskDataset(split_frames['test'], tokenizer, MULTILABEL_TARGETS, INTENSITY_TARGETS), batch_size=BATCH_SIZE)
# val_predictions = predict(model, val_loader)
# test_predictions = predict(model, test_loader)
# thresholds, multi_metrics, regression_metrics = final_evaluate(val_predictions, test_predictions)


## 14. Error Analysis Tables

These tables are where most useful model improvements come from: look at high-confidence false positives, high-confidence false negatives, and large intensity errors.


In [ ]:
def build_prediction_table(split_df, predictions, multilabel_cols, regression_cols, thresholds):
    result = split_df[['question_code'] + [col for col in ['key'] if col in split_df.columns] + TEXT_COLS].copy()
    for i, label in enumerate(multilabel_cols):
        result[f'{label}_true_soft'] = predictions['multilabel_targets'][:, i]
        result[f'{label}_score'] = predictions['multilabel_probs'][:, i]
        result[f'{label}_true_binary'] = (predictions['multilabel_targets'][:, i] >= LABEL_THRESHOLD).astype(int)
        result[f'{label}_pred_binary'] = (predictions['multilabel_probs'][:, i] >= thresholds.get(label, 0.5)).astype(int)
        result[f'{label}_mask'] = predictions['multilabel_mask'][:, i]
    for i, target in enumerate(regression_cols):
        result[f'{target}_true'] = predictions['regression_targets'][:, i]
        result[f'{target}_pred'] = predictions['regression_values'][:, i]
        result[f'{target}_abs_error'] = np.abs(predictions['regression_targets'][:, i] - predictions['regression_values'][:, i])
        result[f'{target}_mask'] = predictions['regression_mask'][:, i]
    return result


def inspect_label_errors(pred_table, label, n=10):
    fp = pred_table[(pred_table[f'{label}_mask'] == 1) & (pred_table[f'{label}_true_binary'] == 0) & (pred_table[f'{label}_pred_binary'] == 1)]
    fn = pred_table[(pred_table[f'{label}_mask'] == 1) & (pred_table[f'{label}_true_binary'] == 1) & (pred_table[f'{label}_pred_binary'] == 0)]
    print(f'False positives for {label}')
    display(fp.sort_values(f'{label}_score', ascending=False).head(n))
    print(f'False negatives for {label}')
    display(fn.sort_values(f'{label}_score', ascending=True).head(n))


def inspect_regression_errors(pred_table, target, n=10):
    subset = pred_table[pred_table[f'{target}_mask'] == 1].sort_values(f'{target}_abs_error', ascending=False)
    display(subset.head(n))


## 15. Ablation Plan

Run these comparisons before deciding the architecture helped:

1. **Existing LLM baseline** from `llm_model_comparison_baseline.ipynb`.
2. **Encoder + strategy multi-label head only**: current task, trainable head.
3. **Encoder + emotion/state multi-label head only**: tests whether affect labels are learnable.
4. **Encoder + intensity regression only**: tests whether graded constructs are learnable.
5. **Full multitask model**: shared encoder, all heads.
6. **Full model without weak labels** if using lexicon/LLM-generated targets.
7. **Full model with grouped split** vs random split to quantify leakage sensitivity.

Keep the winner based on validation, then run one final test evaluation with frozen thresholds.


## 16. Reporting Checklist

Use this checklist in the write-up/results table:

- split method and leakage unit
- number of rows and valid labels per target
- positive rate per multi-label class
- macro F1, micro F1, samples F1, Jaccard, Hamming loss
- per-label precision/recall/F1/AUPRC
- validation-tuned thresholds
- MAE/RMSE and within-tolerance accuracy per intensity item
- safety-sensitive false negative counts
- calibration/Brier score for probability outputs
- ablation table
- examples of high-confidence errors
- human evaluation protocol for generated reappraisals
